## Travel Plan Machine

> **Tourist attractions scraper** 

# Section 1: design the scraper (50 points)

1. Choose a location on Expedia (https://www.expedia.com) and scrape the tourist attractions from that place.
    - For example: Click Things to do on the top menu, and then type the place name Tokyo, Japan → Top 10 things to do → See all (https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F&d1=2024-03-16&d2=2024-03-17&endDate=2024-03-17&filter.seeAll=true&location=Tokyo%20%28and%20vicinity%29%2C%20Tokyo%20Prefecture%2C%20Japan&rid=179900&sort=RECOMMENDED&startDate=2024-03-16&swp=on)
    - <span style="color:red"><b>Notice: each of you should think of a different location, the possibility of getting identical data is low. You should avoid having higher similar syntax or data. Identical syntax and data will be considered plagiarism.</b></span>
2. Collect at least 70 attractions (around 2 pages, 50 on each page), **less than 70 attractions, will get 10 points deducted.**
3. The dataset should include the following details: `{"Attraction_name":[], "Company":[], "Rating":[], "Reviews":[], "Ticket_price":[],"Free_cancellation":[], "Overview":[], "Attraction_page":[]}` **Please use the example outputs as the reference for the content of each column.** 
4. Save your data in a Pandas dataframe.
5. Use df.info() to display the data information.


<span style="color:red"><b>Hints:</b></span>
1. If you encounter a **captcha verification** prompting you to complete a puzzle to prove that you are not a robot, you can solve the puzzle directly on the **driver browser page.** (Use Selenium as a regular browser). Completing one verification should allow you to access the page at least 10 times with the same IP address.
2. If the data does not load properly, for example, if you can't see 100 attractions, you can refresh the driver or close it and reopen the URL. Setting a longer waiting time may help the data load correctly.

In [2]:
# Set up the environment
import time
import pandas as pd
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait as wait
from selenium.common.exceptions import NoSuchElementException

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [3]:
# create Chromeoptions instance 
options = webdriver.ChromeOptions() 
 
# adding argument to disable the AutomationControlled flag 
options.add_argument("--disable-blink-features=AutomationControlled") 
 
# exclude the collection of enable-automation switches 
options.add_experimental_option("excludeSwitches", ["enable-automation"]) 
 
# turn-off userAutomationExtension 
options.add_experimental_option("useAutomationExtension", False) 
 
# setting the driver path and requesting a page 
driver = webdriver.Chrome(options=options) 

# changing the property of the navigator value for webdriver to undefined 
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})") 

I have taken the URL of the top best attractions (9+ rating) in New York between October 28 and October 29.

In [4]:
# DO NOT test the Expedia page too many times on the driver browser, you will easily get blocked
# If you get blocked, clear and delete all of the Cache, Cookies, and Browsing histories about Expedia, 
# and use a VPN to change your IP address if necessary -- You can Google free VPN (many VPN software have a free trial)
# Example: Paris, France (change to your chosen city)
driver.implicitly_wait(10)
url = "https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F=&challengeReferer=noref&d1=2025-10-28&d2=2025-10-29&endDate=2025-10-29&filter.reviewScore=GT4_5&filter.seeAll=true&location=New%20York%20%28and%20vicinity%29%2C%20New%20York%2C%20United%20States%20of%20America&regionType=&rid=178293&selectedId=&slimSearchKeyword=&sort=RECOMMENDED&startDate=2025-10-28&swp=on"
driver.get(url)
time.sleep(30)  # Wait for page and attractions to load

In [5]:
links = []
try:
    # Find all attraction links on the page in a list
    elements = driver.find_elements("xpath", '//div[@class="uitk-card uitk-card-roundcorner-all uitk-card-has-border uitk-layout-grid-item uitk-card-has-primary-theme"]/a')
    # extract all links from the elements list to links
    for elem in elements:
        link = elem.get_attribute('href')
        if link:
            links.append(link)
except:
    links.append("No Link")

In [6]:
len(links)

100

We have collected links of 100 attractions in New York.

Let’s check our links once by printing them here to validate that we have collected the correct and working links by navigating to those links manually.

In [7]:
links[:3]

['https://www.expedia.com/things-to-do/summit-one-vanderbilt-experience-tickets.a44823095.activity-details?&rid=178293&location=New+York+%28and+vicinity%29%2C+New+York%2C+United+States+of+America&startDate=2025-10-28&endDate=2025-10-29&sort=RECOMMENDED&selectedId=&filter.reviewScore=GT4_5&filter.seeAll=true&swp=on',
 'https://www.expedia.com/things-to-do/statue-of-liberty-ellis-island-tour-all-options.a269114.activity-details?&rid=178293&location=New+York+%28and+vicinity%29%2C+New+York%2C+United+States+of+America&startDate=2025-10-28&endDate=2025-10-29&sort=RECOMMENDED&selectedId=&filter.reviewScore=GT4_5&filter.seeAll=true&swp=on',
 'https://www.expedia.com/things-to-do/new-york-citypass-experience-5-must-see-attractions.a184465.activity-details?&rid=178293&location=New+York+%28and+vicinity%29%2C+New+York%2C+United+States+of+America&startDate=2025-10-28&endDate=2025-10-29&sort=RECOMMENDED&selectedId=&filter.reviewScore=GT4_5&filter.seeAll=true&swp=on']

When I'm trying to extract data for more links at a time, it’s taking a lot of time to complete the task, and I'm getting an error, Timed Out. So I will be extracting the data in chunks (25 links each time) to avoid getting errors and for successful data collection.

### Collecting 0-24 Attractions Data

In [10]:
# This is an example output of Things to do in Tokyo
# link = "https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F&d1=2024-03-16&d2=2024-03-17&endDate=2024-03-17&filter.seeAll=true&location=Tokyo%20%28and%20vicinity%29%2C%20Tokyo%20Prefecture%2C%20Japan&rid=179900&sort=RECOMMENDED&startDate=2024-03-16&swp=on"

# YOUR OUTPUT SHOULD DIFFER FROM MINE

# loop and open the attraction link one by one
# pause longer time > 25s to have enough time if captcha verification pops up

attra_dic = {
    "Attraction_name": [],
    "Company": [],
    "Rating": [],
    "Reviews": [],
    "Ticket_price": [],
    "Free_cancellation": [],
    "Overview": [],
    "Location": [],
    "Meeting_point": [],
    "Attraction_page": []
}

for link in links[:25]:
    driver.get(link)
    time.sleep(10)

    # Attraction Name
    try:
        name = driver.find_element('xpath', '//h4[@class="uitk-heading uitk-heading-4 uitk-type-style-headline-large"]').get_attribute('textContent').strip()
    except Exception:
        name = "No Name"

    # Company
    try:
        company = driver.find_element('xpath', '//div[@class="uitk-text uitk-type-300 uitk-text-white-space-pre-line uitk-text-default-theme uitk-spacing uitk-spacing-margin-blockstart-two"]').get_attribute('textContent').strip()
    except Exception:
        company = "No Company"

    # Rating
    try:
        rating = driver.find_element('xpath', '(//span[@class="uitk-badge-base-text"])[1]').get_attribute('textContent').strip()
    except Exception:
        rating = "No Ratings"

    # Reviews
    try:
        review = driver.find_element('xpath', '//button[@class="uitk-link uitk-spacing uitk-spacing-margin-blockstart-one uitk-layout-flex-item uitk-link-align-left uitk-link-layout-default uitk-link-medium"]').get_attribute('textContent').strip()
    except Exception:
        review = "No Reviews"

    # Ticket Price
    try:
        price = driver.find_element('xpath', '(//span[@class="uitk-lockup-price"])[1]').get_attribute('textContent').strip()
    except Exception:
        price = "No Price"

    # Free Cancellation
    try:
        cancellation = driver.find_element('xpath', '//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-layout-grid uitk-layout-grid-has-columns uitk-layout-grid-has-columns-by-medium uitk-layout-grid-has-columns-by-large uitk-spacing uitk-spacing-margin-blockstart-two"]/li[1]').get_attribute('textContent').strip()
    except Exception:
        cancellation = "No Cancellation Details"

    # Overview
    try:
        overview = driver.find_element('xpath', '(//div[@class="uitk-layout-flex-item uitk-layout-flex-item-flex-grow-1"])[2]').get_attribute('textContent').strip()
    except Exception:
        overview = "No Overview"

    # Location
    try:
        location = driver.find_element('xpath', '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[1]/li[1]').get_attribute('textContent').strip()
    except Exception:
        location = "No Location"

    # Meeting Point
    try:
        meeting_point = driver.find_element('xpath', '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[2]/li[1]').get_attribute('textContent').strip()
    except Exception:
        meeting_point = "No Meeting Point"

    # Attraction Page (current URL)
    attraction_page = link

    # Add to dictionary
    attra_dic["Attraction_name"].append(name)
    attra_dic["Company"].append(company)
    attra_dic["Rating"].append(rating)
    attra_dic["Reviews"].append(review)
    attra_dic["Ticket_price"].append(price)
    attra_dic["Free_cancellation"].append(cancellation)
    attra_dic["Overview"].append(overview)
    attra_dic["Location"].append(location)
    attra_dic["Meeting_point"].append(meeting_point)
    attra_dic["Attraction_page"].append(attraction_page)


### Collecting 25-49 Attractions Data

In [11]:
# This is an example output of Things to do in Tokyo
# link = "https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F&d1=2024-03-16&d2=2024-03-17&endDate=2024-03-17&filter.seeAll=true&location=Tokyo%20%28and%20vicinity%29%2C%20Tokyo%20Prefecture%2C%20Japan&rid=179900&sort=RECOMMENDED&startDate=2024-03-16&swp=on"

# YOUR OUTPUT SHOULD DIFFER FROM MINE

# loop and open the attraction link one by one
# pause longer time > 25s to have enough time if captcha verification pops up

for link in links[25:50]:
    driver.get(link)
    time.sleep(10)

    # Attraction Name
    try:
        name = driver.find_element('xpath', '//h4[@class="uitk-heading uitk-heading-4 uitk-type-style-headline-large"]').get_attribute('textContent').strip()
    except Exception:
        name = "No Name"

    # Company
    try:
        company = driver.find_element('xpath', '//div[@class="uitk-text uitk-type-300 uitk-text-white-space-pre-line uitk-text-default-theme uitk-spacing uitk-spacing-margin-blockstart-two"]').get_attribute('textContent').strip()
    except Exception:
        company = "No Company"

    # Rating
    try:
        rating = driver.find_element('xpath', '(//span[@class="uitk-badge-base-text"])[1]').get_attribute('textContent').strip()
    except Exception:
        rating = "No Ratings"

    # Reviews
    try:
        review = driver.find_element('xpath', '//button[@class="uitk-link uitk-spacing uitk-spacing-margin-blockstart-one uitk-layout-flex-item uitk-link-align-left uitk-link-layout-default uitk-link-medium"]').get_attribute('textContent').strip()
    except Exception:
        review = "No Reviews"

    # Ticket Price
    try:
        price = driver.find_element('xpath', '(//span[@class="uitk-lockup-price"])[1]').get_attribute('textContent').strip()
    except Exception:
        price = "No Price"

    # Free Cancellation
    try:
        cancellation = driver.find_element('xpath', '//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-layout-grid uitk-layout-grid-has-columns uitk-layout-grid-has-columns-by-medium uitk-layout-grid-has-columns-by-large uitk-spacing uitk-spacing-margin-blockstart-two"]/li[1]').get_attribute('textContent').strip()
    except Exception:
        cancellation = "No Cancellation Details"

    # Overview
    try:
        overview = driver.find_element('xpath', '(//div[@class="uitk-layout-flex-item uitk-layout-flex-item-flex-grow-1"])[2]').get_attribute('textContent').strip()
    except Exception:
        overview = "No Overview"

    # Location
    try:
        location = driver.find_element('xpath', '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[1]/li[1]').get_attribute('textContent').strip()
    except Exception:
        location = "No Location"

    # Meeting Point
    try:
        meeting_point = driver.find_element('xpath', '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[2]/li[1]').get_attribute('textContent').strip()
    except Exception:
        meeting_point = "No Meeting Point"

    # Attraction Page (current URL)
    attraction_page = link

    # Add to dictionary
    attra_dic["Attraction_name"].append(name)
    attra_dic["Company"].append(company)
    attra_dic["Rating"].append(rating)
    attra_dic["Reviews"].append(review)
    attra_dic["Ticket_price"].append(price)
    attra_dic["Free_cancellation"].append(cancellation)
    attra_dic["Overview"].append(overview)
    attra_dic["Location"].append(location)
    attra_dic["Meeting_point"].append(meeting_point)
    attra_dic["Attraction_page"].append(attraction_page)


### Collecting 50-74 Attractions Data

In [12]:
# This is an example output of Things to do in Tokyo
# link = "https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F&d1=2024-03-16&d2=2024-03-17&endDate=2024-03-17&filter.seeAll=true&location=Tokyo%20%28and%20vicinity%29%2C%20Tokyo%20Prefecture%2C%20Japan&rid=179900&sort=RECOMMENDED&startDate=2024-03-16&swp=on"

# YOUR OUTPUT SHOULD DIFFER FROM MINE

# loop and open the attraction link one by one
# pause longer time > 25s to have enough time if captcha verification pops up

for link in links[50:75]:
    driver.get(link)
    time.sleep(10)

    # Attraction Name
    try:
        name = driver.find_element('xpath', '//h4[@class="uitk-heading uitk-heading-4 uitk-type-style-headline-large"]').get_attribute('textContent').strip()
    except Exception:
        name = "No Name"

    # Company
    try:
        company = driver.find_element('xpath', '//div[@class="uitk-text uitk-type-300 uitk-text-white-space-pre-line uitk-text-default-theme uitk-spacing uitk-spacing-margin-blockstart-two"]').get_attribute('textContent').strip()
    except Exception:
        company = "No Company"

    # Rating
    try:
        rating = driver.find_element('xpath', '(//span[@class="uitk-badge-base-text"])[1]').get_attribute('textContent').strip()
    except Exception:
        rating = "No Ratings"

    # Reviews
    try:
        review = driver.find_element('xpath', '//button[@class="uitk-link uitk-spacing uitk-spacing-margin-blockstart-one uitk-layout-flex-item uitk-link-align-left uitk-link-layout-default uitk-link-medium"]').get_attribute('textContent').strip()
    except Exception:
        review = "No Reviews"

    # Ticket Price
    try:
        price = driver.find_element('xpath', '(//span[@class="uitk-lockup-price"])[1]').get_attribute('textContent').strip()
    except Exception:
        price = "No Price"

    # Free Cancellation
    try:
        cancellation = driver.find_element('xpath', '//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-layout-grid uitk-layout-grid-has-columns uitk-layout-grid-has-columns-by-medium uitk-layout-grid-has-columns-by-large uitk-spacing uitk-spacing-margin-blockstart-two"]/li[1]').get_attribute('textContent').strip()
    except Exception:
        cancellation = "No Cancellation Details"

    # Overview
    try:
        overview = driver.find_element('xpath', '(//div[@class="uitk-layout-flex-item uitk-layout-flex-item-flex-grow-1"])[2]').get_attribute('textContent').strip()
    except Exception:
        overview = "No Overview"

    # Location
    try:
        location = driver.find_element('xpath', '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[1]/li[1]').get_attribute('textContent').strip()
    except Exception:
        location = "No Location"

    # Meeting Point
    try:
        meeting_point = driver.find_element('xpath', '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[2]/li[1]').get_attribute('textContent').strip()
    except Exception:
        meeting_point = "No Meeting Point"

    # Attraction Page (current URL)
    attraction_page = link

    # Add to dictionary
    attra_dic["Attraction_name"].append(name)
    attra_dic["Company"].append(company)
    attra_dic["Rating"].append(rating)
    attra_dic["Reviews"].append(review)
    attra_dic["Ticket_price"].append(price)
    attra_dic["Free_cancellation"].append(cancellation)
    attra_dic["Overview"].append(overview)
    attra_dic["Location"].append(location)
    attra_dic["Meeting_point"].append(meeting_point)
    attra_dic["Attraction_page"].append(attraction_page)


Let’s convert the data we collected in the dictionary, using lists for each variable, to a pandas dataframe for analysis.

In [13]:
# save the data to a DataFrame
# if you get restricted at some time points, you may likely get many N/A values like I got here
# which is okay, and continue the analysis
df = pd.DataFrame(attra_dic)
df

,Attraction_name,Company,Rating,Reviews,Ticket_price,Free_cancellation,Overview,Location,Meeting_point,Attraction_page
0,SUMMIT One Vanderbilt Experience Tickets,By SUMMIT One Vanderbilt,9.2,809 verified reviewsMore information about rev...,$48,Free cancellation available,OverviewMost immersive observation deck - an I...,45 East 42nd Street,45 East 42nd Street,https://www.expedia.com/things-to-do/summit-on...
1,Statue of Liberty & Ellis Island Tour: All Opt...,By ExperienceFirst,9.4,"1,829 verified reviewsMore information about r...",$69,Free cancellation available,OverviewEngaging guided tour to the Statue of ...,"10004, New York, New York, United States of Am...",Battery Park,https://www.expedia.com/things-to-do/statue-of...
2,New York CityPASS®: Experience 5 must-see attr...,By CityPASS,9.0,"2,106 verified reviewsMore information about r...",$154,Free cancellation available,OverviewSave time and money with one simple pu...,"New York, New York, United States of America",1071 5th Avenue,https://www.expedia.com/things-to-do/new-york-...
3,The Lion King On Broadway,By The Lion King on Broadway,9.4,"1,452 verified reviewsMore information about r...",$160,2h 30m,OverviewThe highest-grossing Broadway producti...,200 West 45th Street,200 West 45th Street,https://www.expedia.com/things-to-do/the-lion-...
4,One World Observatory Tickets,By One World Observatory,9.0,"1,225 verified reviewsMore information about r...",$31,Free cancellation available,OverviewSpectacular views from Western Hemisph...,117 West Street,117 West Street,https://www.expedia.com/things-to-do/one-world...
...,...,...,...,...,...,...,...,...,...,...
70,Guided Standard Central Park Horse Carriage Ri...,By NYCAdventures,9.0,9 verified reviewsMore information about revie...,$185,Free cancellation available,OverviewExplore Central Park on a beautiful ho...,78 Central Park South,78 Central Park South,https://www.expedia.com/things-to-do/guided-st...
71,Skip-the-Line Metropolitan Museum of Art Highl...,By ExperienceFirst,10,4 verified reviewsMore information about revie...,$65,Free cancellation available,OverviewConnect with the art on a personal lev...,1000 5th Ave,1000 5th Ave,https://www.expedia.com/things-to-do/skip-the-...
72,Half-Day Sightseeing Tour of Brooklyn,By Tours with Dani,9.8,9 Viator reviewsMore information about reviews...,$59,Free cancellation available,"Activity locationBrooklyn Bridge10038, New Yor...","10038, New York City, New York, United States",164 Bedford Avenue,https://www.expedia.com/things-to-do/half-day-...
73,Marvel Universe in New York Self-Guided Audio ...,By WeGo Trip,10,1 verified reviewMore information about review...,$6,1h 30m,"OverviewLearn about Marvel series, cartoons, a...",135 West 50th Street,135 West 50th Street,https://www.expedia.com/things-to-do/marvel-un...


Now, I will clean the data:

1. Convert `Rating` to float type.
2. Replace the `$` from `Ticket_Price` and convert it to float.
3. Extract the numerical part from `Reviews` and convert it to float.
4. Clean the `Company` column.

In [14]:
df["Rating"] = df["Rating"].astype(float)
df["Ticket_price"] = df["Ticket_price"].str.replace("$", "", regex=False).astype(float)
df["Reviews"] = df["Reviews"].str.replace(",", "", regex=False).str.extract(r"(\d+)").astype(float)
df["Company"] = df["Company"].str.replace(r"^By\s*", "", regex=True).str.strip()

I will save the data to a CSV for future reference, to ignore the part to collect the same data again, which leads to website blocking me.

In [15]:
df.to_csv("attractions_raw.csv")

In [16]:
# display the basic information of the data
# YOUR OUTPUT SHOULD DIFFER FROM MINE
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Attraction_name    75 non-null     object 
 1   Company            75 non-null     object 
 2   Rating             75 non-null     float64
 3   Reviews            75 non-null     float64
 4   Ticket_price       75 non-null     float64
 5   Free_cancellation  75 non-null     object 
 6   Overview           75 non-null     object 
 7   Location           75 non-null     object 
 8   Meeting_point      75 non-null     object 
 9   Attraction_page    75 non-null     object 
dtypes: float64(3), object(7)
memory usage: 6.0+ KB


I hae collected 75 attractions details with no null values.

# Section 2: Basic data analysis (40 points)

**Q1.** In your own words, describe the difference between a `while loop` and a `for loop` in Python.
When working on a web scraping project, in what situations would you use each type of loop? (5 points)

A `for` loop is used to loop through a known number of times, or to iterate through a data structure such as a list or a dictionary. For example, we can use a  for loop to iterate through the list of URLs to the websites, like we did above to iterate through the `links` list, in which we have stored all the URLs to the attractions in New York. It is to be used when we know exactly how many times we need to iterate.

The `while` loop will run until the condition we specify is `True` without knowing how many times it will run beforehand. It is useful for dynamic repetitions when you don't know how many times you need to repeat the loop, but you see the condition till when you need to run it.

In web scraping:

* `for` loop can be used to iterate through the lists or dictionaries while extracting data.
* `while` lopps can be used to click the `next` button to load all the pages until there is no `next` page available (to the end.

**Q2.** Drop the missing values from "Ticket_price" column. (3 points)
   - If you don't have missing values for "Ticket_price", just list the function (use comments) that will be used for excuting this step.

In [17]:
# after dropping all the null values from the Ticket_price, I got 55 data points here
# YOUR OUTPUT SHOULD DIFFER FROM MINE
df.dropna(subset=['Ticket_price'], inplace=True)

**Q3.** Display the 15th row in the dataframe. (3 points)

In [18]:
df.iloc[14]

Attraction_name        Circle Line: Statue of Liberty at Sunset Cruise
Company                                            New York Water Taxi
Rating                                                             9.8
Reviews                                                            6.0
Ticket_price                                                      32.0
Free_cancellation                          Free cancellation available
Overview             OverviewThe Statue of Liberty at SunsetThe Man...
Location                                               89 South Street
Meeting_point                                          89 South Street
Attraction_page      https://www.expedia.com/things-to-do/circle-li...
Name: 14, dtype: object

**Q4.** Fill the null values in "Rating" with the number 0. (5 points)
- If you don't have missing values for "Rating", just list the function (use comments) that will be used for executing this step.

In [19]:
# YOUR OUTPUT SHOULD DIFFER FROM MINE
# collenting the code as I don't have missing values in the column 'Rating'
# df['Rating'].fillna(0, inplace=True)
df.head()

,Attraction_name,Company,Rating,Reviews,Ticket_price,Free_cancellation,Overview,Location,Meeting_point,Attraction_page
0,SUMMIT One Vanderbilt Experience Tickets,SUMMIT One Vanderbilt,9.2,809.0,48.0,Free cancellation available,OverviewMost immersive observation deck - an I...,45 East 42nd Street,45 East 42nd Street,https://www.expedia.com/things-to-do/summit-on...
1,Statue of Liberty & Ellis Island Tour: All Opt...,ExperienceFirst,9.4,1829.0,69.0,Free cancellation available,OverviewEngaging guided tour to the Statue of ...,"10004, New York, New York, United States of Am...",Battery Park,https://www.expedia.com/things-to-do/statue-of...
2,New York CityPASS®: Experience 5 must-see attr...,CityPASS,9.0,2106.0,154.0,Free cancellation available,OverviewSave time and money with one simple pu...,"New York, New York, United States of America",1071 5th Avenue,https://www.expedia.com/things-to-do/new-york-...
3,The Lion King On Broadway,The Lion King on Broadway,9.4,1452.0,160.0,2h 30m,OverviewThe highest-grossing Broadway producti...,200 West 45th Street,200 West 45th Street,https://www.expedia.com/things-to-do/the-lion-...
4,One World Observatory Tickets,One World Observatory,9.0,1225.0,31.0,Free cancellation available,OverviewSpectacular views from Western Hemisph...,117 West Street,117 West Street,https://www.expedia.com/things-to-do/one-world...


**Q5.** Display the rating range of the dataset and select the top rating attractions (Rating scores >= 9). (5 points)

In [20]:
# display the Rating range
df["Rating"].unique()

array([ 9.2,  9.4,  9. ,  9.6, 10. ,  9.8])

The ratings will be above 9 because I’ve scraped the top destinations, which usually have ratings exceeding 9 (in this case).

In [21]:
# select the top-rated attractions
# YOUR OUTPUT SHOULD DIFFER FROM MINE
df[df['Rating'] >= 9]

,Attraction_name,Company,Rating,Reviews,Ticket_price,Free_cancellation,Overview,Location,Meeting_point,Attraction_page
0,SUMMIT One Vanderbilt Experience Tickets,SUMMIT One Vanderbilt,9.2,809.0,48.0,Free cancellation available,OverviewMost immersive observation deck - an I...,45 East 42nd Street,45 East 42nd Street,https://www.expedia.com/things-to-do/summit-on...
1,Statue of Liberty & Ellis Island Tour: All Opt...,ExperienceFirst,9.4,1829.0,69.0,Free cancellation available,OverviewEngaging guided tour to the Statue of ...,"10004, New York, New York, United States of Am...",Battery Park,https://www.expedia.com/things-to-do/statue-of...
2,New York CityPASS®: Experience 5 must-see attr...,CityPASS,9.0,2106.0,154.0,Free cancellation available,OverviewSave time and money with one simple pu...,"New York, New York, United States of America",1071 5th Avenue,https://www.expedia.com/things-to-do/new-york-...
3,The Lion King On Broadway,The Lion King on Broadway,9.4,1452.0,160.0,2h 30m,OverviewThe highest-grossing Broadway producti...,200 West 45th Street,200 West 45th Street,https://www.expedia.com/things-to-do/the-lion-...
4,One World Observatory Tickets,One World Observatory,9.0,1225.0,31.0,Free cancellation available,OverviewSpectacular views from Western Hemisph...,117 West Street,117 West Street,https://www.expedia.com/things-to-do/one-world...
...,...,...,...,...,...,...,...,...,...,...
70,Guided Standard Central Park Horse Carriage Ri...,NYCAdventures,9.0,9.0,185.0,Free cancellation available,OverviewExplore Central Park on a beautiful ho...,78 Central Park South,78 Central Park South,https://www.expedia.com/things-to-do/guided-st...
71,Skip-the-Line Metropolitan Museum of Art Highl...,ExperienceFirst,10.0,4.0,65.0,Free cancellation available,OverviewConnect with the art on a personal lev...,1000 5th Ave,1000 5th Ave,https://www.expedia.com/things-to-do/skip-the-...
72,Half-Day Sightseeing Tour of Brooklyn,Tours with Dani,9.8,9.0,59.0,Free cancellation available,"Activity locationBrooklyn Bridge10038, New Yor...","10038, New York City, New York, United States",164 Bedford Avenue,https://www.expedia.com/things-to-do/half-day-...
73,Marvel Universe in New York Self-Guided Audio ...,WeGo Trip,10.0,1.0,6.0,1h 30m,"OverviewLearn about Marvel series, cartoons, a...",135 West 50th Street,135 West 50th Street,https://www.expedia.com/things-to-do/marvel-un...


**Q6.** Selecting the "full-day" trips from the attraction titles. (2 points)

In [22]:
df["Attraction_name"].unique()

array(['SUMMIT One Vanderbilt Experience Tickets',
       'Statue of Liberty & Ellis Island Tour: All Options',
       'New York CityPASS®: Experience 5 must-see attractions',
       'The Lion King On Broadway', 'One World Observatory Tickets',
       'Aladdin on Broadway', 'Wicked On Broadway',
       'Central Park Pedicab Tour– Top Highlights',
       'Color Factory NYC Ticket', 'Intrepid Museum Admission Ticket',
       'Fully Guided Statue of Liberty Tour with Ellis Island',
       'Brooklyn Small Group Food Tour: Local Bites & Neighborhood Stories',
       'Circle Line: 50-Minute Statue of Liberty Super Express Sightseeing Cruise',
       'Circle Line: New York Landmarks Cruise',
       'Circle Line: Statue of Liberty at Sunset Cruise',
       'New York: Unlimited USA Internet with eSIM Mobile Data Plan',
       'Edge Observation Deck at Hudson Yards',
       'NYC: Museum of Modern Art (MoMA) Entry Ticket',
       'Brooklyn Nets Basketball Game at Barclays Center',
       'All Day

In [23]:
# YOUR OUTPUT SHOULD DIFFER FROM MINE
df[df["Attraction_name"].str.contains("full-day", case=False, na=False)]

,Attraction_name,Company,Rating,Reviews,Ticket_price,Free_cancellation,Overview,Location,Meeting_point,Attraction_page


There might be some full-day trips in New York, but none of them are mentioned in Expedia’s Attraction titles. Therefore, based on the available data, there is no information about full-day trips.

**Q7.** Display the top 10 most expansive places in the dataframe. (7 points)

In [24]:
# hint: To retrieve numbers from text, you can use the regular expression [^0-9] to extract numeric values.
expensive_10 = df.sort_values("Ticket_price", ascending=False).head(10)
expensive_10

,Attraction_name,Company,Rating,Reviews,Ticket_price,Free_cancellation,Overview,Location,Meeting_point,Attraction_page
36,The Empire Helicopter Tour of New York,Charm Aviation NYC,9.6,19.0,429.0,Free cancellation available,Activity location6 East River Piers6 East Rive...,6 East River Piers,6 East River Piers,https://www.expedia.com/things-to-do/the-empir...
47,The Big Apple Helicopter Tour of New York City,Charm Aviation NYC,9.0,189.0,319.0,Free cancellation available,Activity location6 East River Piers6 East Rive...,6 East River Piers,6 East River Piers,https://www.expedia.com/things-to-do/the-big-a...
50,Big City Helicopter Tour,Zip Aviation,9.4,24.0,279.0,Free cancellation available,OverviewSpectacular views of the Big Apple as ...,6 East River Piers,6 East River Piers,https://www.expedia.com/things-to-do/big-city-...
18,Brooklyn Nets Basketball Game at Barclays Center,Sports Where I Am,9.6,3.0,200.0,3h,OverviewSee a Brooklyn Nets NBA game in person...,620 Atlantic Avenue,620 Atlantic Avenue,https://www.expedia.com/things-to-do/brooklyn-...
30,Washington D.C 1-Day City Tour from New York City,Jupiter Legend Corporation,10.0,1.0,199.0,Free cancellation available,OverviewVisit Washington D.C.’s top landmarks ...,U.S. Capitol Visitor Center,"06:45 Departure Red Lobster, Times Square; 5 T...",https://www.expedia.com/things-to-do/best-amis...
51,Official Exclusive VIP Horse Carriage Ride in ...,NYC HORSE AND CARRIAGE RIDE,10.0,5.0,195.0,Free cancellation available,OverviewExplore Central Park's gems on a guide...,No Location,No Meeting Point,https://www.expedia.com/things-to-do/official-...
70,Guided Standard Central Park Horse Carriage Ri...,NYCAdventures,9.0,9.0,185.0,Free cancellation available,OverviewExplore Central Park on a beautiful ho...,78 Central Park South,78 Central Park South,https://www.expedia.com/things-to-do/guided-st...
25,MJ The Musical on Broadway,MJ The Musical,9.8,51.0,173.0,2h 30m,OverviewThe life story of Michael Jackson arri...,250 West 52nd Street,250 West 52nd Street,https://www.expedia.com/things-to-do/mj-the-mu...
3,The Lion King On Broadway,The Lion King on Broadway,9.4,1452.0,160.0,2h 30m,OverviewThe highest-grossing Broadway producti...,200 West 45th Street,200 West 45th Street,https://www.expedia.com/things-to-do/the-lion-...
2,New York CityPASS®: Experience 5 must-see attr...,CityPASS,9.0,2106.0,154.0,Free cancellation available,OverviewSave time and money with one simple pu...,"New York, New York, United States of America",1071 5th Avenue,https://www.expedia.com/things-to-do/new-york-...


**Q8.** If you have a budget of 10,000 dollars for traveling to the place you selected, will it cover the attractions listed above (top 10 most expensive attractions)? (2 points)

In [25]:
# use the function to calculate the total sum of the prices of the top 10 attractions
total_cost = expensive_10["Ticket_price"].sum()
print("Total cost of top 10 attractions:", total_cost)
budget = 10000

if total_cost <= budget:
    print("Yes, $10,000 budget will cover all top 10 attractions.")
else:
    print("No, $10,000 budget is not enough.")

Total cost of top 10 attractions: 2293.0
Yes, $10,000 budget will cover all top 10 attractions.


**Q9.** Among the top 10 most expensive attractions, which one has the most number of reviews? (2 point)

In [26]:
# Sort by Review in descending order
expensive_10.sort_values(by="Reviews", ascending=False).head(1)

,Attraction_name,Company,Rating,Reviews,Ticket_price,Free_cancellation,Overview,Location,Meeting_point,Attraction_page
2,New York CityPASS®: Experience 5 must-see attr...,CityPASS,9.0,2106.0,154.0,Free cancellation available,OverviewSave time and money with one simple pu...,"New York, New York, United States of America",1071 5th Avenue,https://www.expedia.com/things-to-do/new-york-...


The most expensive attraction is the “Washington D.C. 1-Day City Tour from New York City,” with a ticket price of $199.

We can see this is a full 1-Day tour.

**Q10.** Groupby the Travel agencies and count how many attractions they provide for each agency. (2 points)

In [27]:
# YOUR OUTPUT SHOULD DIFFER FROM MINE
df.groupby("Company")["Attraction_name"].count()

Company
#1 Central Park Pedicab Tour             1
Aladdin on Broadway                      1
American Dream                           1
Attractions4US                           1
Bee Lively Photography                   1
Big City Tourism                         1
Bike Rent NYC                            1
Broadway Inbound US                      3
Charm Aviation NYC                       2
Chicago on Broadway                      1
Circle Line Sightseeing                  2
City Wonders                             4
CityPASS                                 1
Color Factory NYC                        1
Cycle Park NYC                           1
Edge Hudson Yards Observation Deck       1
Empire Vacations                         1
Experience NYC™                          1
ExperienceFirst                          4
ExperienceNYC                            1
Intrepid Air & Space Museum              1
Intrepid Travel                          4
Jupiter Legend Corporation               1
MJ 

**Q11.** If you need to use the entire dataset and analysis to plan your trip, please write down your plans. (4 points)

If I were to use the entire dataset for my analysis and trip planning, I would start by filtering the data systematically. First, I would **sort** the dataset by both `Rating` and `Reviews` to identify the attractions that are most popular and highly rated by visitors. Next, I would **filter** the results to include only those attractions that fit within my **budget range** and offer **free cancellation**, ensuring flexibility in case my plans change. I would also **group** the attractions by their `Company` to see which tour providers consistently deliver high-quality experiences and possibly select multiple attractions from the same provider for convenience or package discounts.

Once I have shortlisted the top attractions, I would extract their `Location` and `Meeting_point` details to optimize the travel route. By analyzing the Loaction, I could plan the **shortest and most efficient path** between attractions—minimizing travel time and maximizing sightseeing opportunities. Additionally, I would consider each attraction’s **operating hours** (checking in the attraction page) to arrange my itinerary in a logical sequence, starting with morning-friendly activities and reserving indoor or evening events for later in the day.

Overall, I would use a combination of sorting, filtering, grouping to create a well-organized, data-driven trip plan that balances cost, convenience, and experience quality.

In [28]:
df.to_csv("attractions.csv")

## Tourist reviews scraper

> design the scraper (30 points)

For each attraction, there are multiple customer reviews. Please design a scraper to get the customer review content. 

**Instructions:**

1. Get 1 page of customer reviews (around 10 reviews) for the first 2 attractions from the above scraper.
2. The dataset should include the following details: `{"Attraction_page":[], "Customer_name":[], "Review_scores":[], "Review_date":[], "Country":[], "Review_text":[]}` **Please use the example outputs as the reference for the content of each column.**
3. Remove all the special characters from the texts. If the review texts don't contain special characters, write the Python function or code snippet you would use to perform this cleaning step.

hint for some xpath:

1) To get all the names of the customers, you can try with`//div[@class="uitk-layout-grid-item"][2]/div/h3[@class="uitk-heading uitk-heading-6 uitk-spacing uitk-spacing-margin-blockstart-two"]`

In [30]:
# This is an example output of Things to do in Tokyo
# link = "https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F&d1=2024-03-16&d2=2024-03-17&endDate=2024-03-17&filter.seeAll=true&location=Tokyo%20%28and%20vicinity%29%2C%20Tokyo%20Prefecture%2C%20Japan&rid=179900&sort=RECOMMENDED&startDate=2024-03-16&swp=on"

# YOUR OUTPUT SHOULD DIFFER FROM MINE

In [31]:
review_dic = {
    "Attraction_page": [],
    "Customer_name": [],
    "Review_scores": [],
    "Review_date": [],
    "Country": [],
    "Review_text": []
}


for link in links[:2]:
    driver.get(link)
    time.sleep(10)  # Wait for reviews to load

    # Find all review containers on this page
    names = driver.find_elements('xpath', '//div[@class="uitk-layout-grid-item"][2]/div/h3[@class="uitk-heading uitk-heading-6 uitk-spacing uitk-spacing-margin-blockstart-two"]')
    scores = driver.find_elements('xpath', '//div[@class="uitk-layout-grid-item"][2]/div/h4[@class="uitk-heading uitk-heading-6"]')
    dates = driver.find_elements('xpath', '//div[@class="uitk-layout-grid-item"][2]/div/div[@class="uitk-text uitk-type-300 uitk-text-default-theme"][2]')
    countries = driver.find_elements('xpath', '//div[@class="uitk-layout-grid-item"][2]/div/div[@class="uitk-text uitk-type-300 uitk-text-default-theme"][3]')
    texts = driver.find_elements('xpath', '//div[@class="uitk-layout-grid-item"][2]/div/div[@class="uitk-expando-peek uitk-spacing uitk-spacing-margin-blockstart-two"]//div[@class="uitk-text overflow-wrap uitk-type-300 uitk-text-default-theme"]')
    for name in names:
        try:
            review_name = name.get_attribute('textContent').strip()
        except:
            review_name = "No Name"
    for score in scores:
        try:
            review_score = score.get_attribute('textContent').strip()
        except:
            review_score = ""
    for date in dates:
        try:
            review_date = date.get_attribute('textContent').strip()
        except:
            review_date = ""
    for country in countries:
        try:
            review_country = country.get_attribute('textContent').strip()
        except:
            review_country = ""
    for text in texts:
        try:
            review_text = text.get_attribute('textContent').strip()
        except:
            review_text = ""

        review_dic['Attraction_page'].append(link)
        review_dic['Customer_name'].append(review_name)
        review_dic['Review_scores'].append(review_score)
        review_dic['Review_date'].append(review_date)
        review_dic['Country'].append(review_country)
        review_dic['Review_text'].append(review_text)


In [32]:
df = pd.DataFrame(review_dic)
df

,Attraction_page,Customer_name,Review_scores,Review_date,Country,Review_text
0,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,It’s was great I really enjoyed the day I visi...
1,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Impressive views over City but thought immersi...
2,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Amazing views. Some of elements felt like the...
3,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,"Great, besides the balloon room would do great..."
4,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Great experience and fun at Summit one. The li...
5,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Summit Vanderbilt was an Amazing experience. ...
6,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,So interactive. The experience is amazing. So ...
7,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Amazing experience! Highly recommended \n360 ...
8,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,The experience was nice it was just too crowde...
9,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,It was a very very wonderful and enjoying expe...


In [33]:
# clean the review texts
df['Review_text'] = (
    df['Review_text']
    .str.replace(r'&#\d+;', '', regex=True)
    .str.replace(r'[^A-Za-z0-9\s.,!?\'"]+', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [34]:
# YOUR OUTPUT SHOULD DIFFER FROM MINE
df

,Attraction_page,Customer_name,Review_scores,Review_date,Country,Review_text
0,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,It s was great I really enjoyed the day I visi...
1,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Impressive views over City but thought immersi...
2,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Amazing views. Some of elements felt like they...
3,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,"Great, besides the balloon room would do great..."
4,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Great experience and fun at Summit one. The li...
5,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Summit Vanderbilt was an Amazing experience. S...
6,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,So interactive. The experience is amazing. So ...
7,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,Amazing experience! Highly recommended 360 spe...
8,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,The experience was nice it was just too crowde...
9,https://www.expedia.com/things-to-do/summit-on...,R G,10.0/10,"Aug 23, 2025",United States,It was a very very wonderful and enjoying expe...


In [35]:
df.to_csv('reviews.csv')

## Excellent! You have finished the mid-term exam!